In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# BLOCK 1 │ Imports & Hardware Setup
# ─────────────────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.amp import autocast, GradScaler
import pandas as pd
import numpy as np
import optuna
from optuna.samplers import TPESampler
import gc
import warnings
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import os
from pathlib import Path
 
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Multi-GPU Setup ──────────────────────────────────────────────────────────
N_GPUS  = min(torch.cuda.device_count(), 2)
PRIMARY = torch.device("cuda:0")
scaler_amp = GradScaler(device='cuda')
 
print(f"🚀 Using {N_GPUS} GPU(s):")
for i in range(N_GPUS):
    p = torch.cuda.get_device_properties(i)
    print(f"   [{i}] {p.name}  ({p.total_memory / 1e9:.1f} GB)")
 
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

In [ ]:
# Configuration and Feature Selection
FEATURES = [
    'precip_1hr [inch]', 
    'precip_max_intensity [inch/hour]', 
    'temp_2m [degF]', 
    'soil_moisture_05cm [m^3/m^3]', 
    'elevation [feet]'
]
TARGET = 'depth_inches'
TV_SPLIT = (0.70, 0.15, 0.15)  # Train / Val / Test proportions

# 1. Identify the current directory (Handles both .py scripts and Jupyter)
try:
    current_location = Path(__file__).resolve().parent
except NameError:
    current_location = Path.cwd().resolve()

# 2. Navigate to the actual Project Root
if current_location.name in ["Finalized_Scripts", "Test_Scripts", "scripts"]:
    PROJECT_ROOT = current_location.parent
else:
    PROJECT_ROOT = current_location

# 3. Define the absolute path to the data
data_dir = PROJECT_ROOT / "Data_Files"
file_name = "rain_influenced_gages.parquet"
file_path = data_dir / file_name

# 4. Safety Check and Data Loading
if not file_path.exists():
    raise FileNotFoundError(f"Target file not found at: {file_path}")

df = pd.read_parquet(file_path)
print(f"Successfully loaded data from: {file_path}")

In [ ]:
# ── Resolve storm identifier column ─────────────────────────────────────────
STORM_COL = None
for candidate in ['storm_id', 'event_id', 'storm', 'event']:
    if candidate in df.columns:
        STORM_COL = candidate
        break
 
if STORM_COL is None:
    if isinstance(df.index, pd.DatetimeIndex):
        gap_seconds = df.index.to_series().diff().dt.total_seconds().fillna(0)
        df['_storm_id'] = (gap_seconds > 6 * 3600).cumsum()
    else:
        CHUNK = 500
        df['_storm_id'] = np.arange(len(df)) // CHUNK
    STORM_COL = '_storm_id'
    print(f"⚠️  No storm ID column found. Inferred {df[STORM_COL].nunique()} events from time-gap criterion (>6 h).")
else:
    print(f"✅ Storm column '{STORM_COL}' — {df[STORM_COL].nunique()} events detected.")

# ── Drop incomplete rows, cast to float32 ───────────────────────────────────
df_clean = df[FEATURES + [TARGET, STORM_COL]].dropna().copy()
df_clean[FEATURES + [TARGET]] = df_clean[FEATURES + [TARGET]].astype('float32')

In [ ]:
# ── Chronological storm-level split ─────────────────────────────────────────
storm_ids = df_clean[STORM_COL].unique()
n_storms  = len(storm_ids)
n_tr      = int(n_storms * TV_SPLIT[0])
n_va      = int(n_storms * TV_SPLIT[1])
 
train_storms = storm_ids[:n_tr]
val_storms   = storm_ids[n_tr : n_tr + n_va]
test_storms  = storm_ids[n_tr + n_va :]
 
train_df = df_clean[df_clean[STORM_COL].isin(train_storms)].copy()
val_df   = df_clean[df_clean[STORM_COL].isin(val_storms)].copy()
test_df  = df_clean[df_clean[STORM_COL].isin(test_storms)].copy()
 
print(f"\n📊 Storm-aware partition (no mid-storm cuts):")
print(f"   Train : {len(train_df):>8,} rows  ({len(train_storms):>4} storms)")
print(f"   Val   : {len(val_df):>8,} rows  ({len(val_storms):>4} storms)")
print(f"   Test  : {len(test_df):>8,} rows  ({len(test_storms):>4} storms)")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# BLOCK 3 │ Scaling & GPU Tensor Push
# ─────────────────────────────────────────────────────────────────────────────
scaler_X = StandardScaler()
scaler_y = StandardScaler()
 
X_tr  = scaler_X.fit_transform(train_df[FEATURES]).astype('float32')
X_val = scaler_X.transform(val_df[FEATURES]).astype('float32')
X_te  = scaler_X.transform(test_df[FEATURES]).astype('float32')
 
y_tr  = scaler_y.fit_transform(train_df[[TARGET]]).astype('float32')
y_val = scaler_y.transform(val_df[[TARGET]]).astype('float32')
 
y_val_raw = val_df[TARGET].values.astype('float32')
y_te_raw  = test_df[TARGET].values.astype('float32')
 
sid_tr  = train_df[STORM_COL].values
sid_val = val_df[STORM_COL].values
sid_te  = test_df[STORM_COL].values
 
X_tr_gpu      = torch.tensor(X_tr,      device=PRIMARY)
y_tr_gpu      = torch.tensor(y_tr,      device=PRIMARY)
X_val_gpu     = torch.tensor(X_val,     device=PRIMARY)
X_te_gpu      = torch.tensor(X_te,      device=PRIMARY)
y_val_raw_gpu = torch.tensor(y_val_raw, device=PRIMARY)
y_te_raw_gpu  = torch.tensor(y_te_raw,  device=PRIMARY)
 
Y_MEAN = torch.tensor(scaler_y.mean_,  device=PRIMARY, dtype=torch.float32)
Y_STD  = torch.tensor(scaler_y.scale_, device=PRIMARY, dtype=torch.float32)
 
def descale(p: torch.Tensor) -> torch.Tensor:
    """Invert standard-scaling on predicted depth."""
    return p * Y_STD + Y_MEAN
 
print(f"✅ Tensors on {PRIMARY}. VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# BLOCK 4 │ LSTM Storm-Safe Window Builder
# ─────────────────────────────────────────────────────────────────────────────
def build_storm_windows(X: np.ndarray, y: np.ndarray,
                        storm_ids: np.ndarray,
                        window: int):
    Xw, yw = [], []
    for sid in np.unique(storm_ids):
        mask = storm_ids == sid
        Xs, ys = X[mask], y[mask]
        n = len(Xs)
        if n <= window:
            continue
        for i in range(n - window):
            Xw.append(Xs[i : i + window])
            yw.append(ys[i + window])
    if len(Xw) == 0:
        return np.empty((0, window, X.shape[1]), dtype='float32'), np.empty((0, 1), dtype='float32')
    return (np.array(Xw, dtype='float32'),
            np.array(yw, dtype='float32').reshape(-1, 1))

_WINDOW_CACHE = {}

def get_windows(split: str, window: int):
    key = (split, window)
    if key not in _WINDOW_CACHE:
        if split == 'train':
            Xw, yw = build_storm_windows(X_tr, y_tr, sid_tr, window)
        elif split == 'val':
            Xw, yw = build_storm_windows(X_val, y_val, sid_val, window)
        else:
            y_te_sc = scaler_y.transform(test_df[[TARGET]]).astype('float32')
            Xw, yw  = build_storm_windows(X_te, y_te_sc, sid_te, window)
        
        _WINDOW_CACHE[key] = (
            torch.tensor(Xw, dtype=torch.float32), 
            torch.tensor(yw, dtype=torch.float32),
        )
    return _WINDOW_CACHE[key]

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# BLOCK 5 │ Model Architectures
# ─────────────────────────────────────────────────────────────────────────────
class ResidualBlock(nn.Module):
    def __init__(self, size: int, dropout: float = 0.1):
        super().__init__()
        self.norm = nn.LayerNorm(size)
        self.fc   = nn.Linear(size, size)
        self.drop = nn.Dropout(dropout)
 
    def forward(self, x):
        return x + self.drop(F.relu(self.fc(self.norm(x))))
 
class SotaANN(nn.Module):
    def __init__(self, input_size: int, hidden_size: int,
                 n_layers: int = 3, dropout: float = 0.1):
        super().__init__()
        self.proj   = nn.Linear(input_size, hidden_size)
        self.blocks = nn.Sequential(
            *[ResidualBlock(hidden_size, dropout) for _ in range(n_layers)]
        )
        self.head = nn.Linear(hidden_size, 1)
 
    def forward(self, x):
        return self.head(self.blocks(F.relu(self.proj(x))))
 
class SotaAttentionLSTM(nn.Module):
    def __init__(self, input_size: int, hidden_size: int,
                 n_layers: int = 2, dropout: float = 0.15):
        super().__init__()
        lstm_drop = dropout if n_layers > 1 else 0.0
        self.lstm = nn.LSTM(input_size, hidden_size,
                            num_layers=n_layers, batch_first=True,
                            bidirectional=True, dropout=lstm_drop)
        self.attn = nn.Linear(hidden_size * 2, 1)
        self.norm = nn.LayerNorm(hidden_size * 2)
        self.head = nn.Linear(hidden_size * 2, 1)
 
    def forward(self, x):
        out, _   = self.lstm(x)                          
        weights  = F.softmax(self.attn(out), dim=1)      
        context  = torch.sum(out * weights, dim=1)        
        return self.head(self.norm(context))
 
def wrap_model(model: nn.Module) -> nn.Module:
    if N_GPUS > 1:
        model = nn.DataParallel(model, device_ids=list(range(N_GPUS)))
    return model.to(PRIMARY)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# BLOCK 6 │ Hydrological Performance Metrics
# ─────────────────────────────────────────────────────────────────────────────
def nse(y_true: torch.Tensor, y_pred: torch.Tensor) -> float:
    num = torch.sum((y_true - y_pred) ** 2)
    den = torch.sum((y_true - y_true.mean()) ** 2) + 1e-9
    return (1 - num / den).item()
 
def kge(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    r     = np.corrcoef(y_true, y_pred)[0, 1]
    alpha = y_pred.std()  / (y_true.std()  + 1e-9)
    beta  = y_pred.mean() / (y_true.mean() + 1e-9)
    return 1 - np.sqrt((r - 1)**2 + (alpha - 1)**2 + (beta - 1)**2)
 
def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
 
def pbias(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(100 * np.sum(y_true - y_pred) / (np.sum(y_true) + 1e-9))
 
def eval_metrics(name: str, y_true_np: np.ndarray, y_pred_np: np.ndarray) -> dict:
    yt = torch.tensor(y_true_np, device=PRIMARY)
    yp = torch.tensor(y_pred_np, device=PRIMARY)
    return {
        'Model': name,
        'NSE':   round(nse(yt, yp), 4),
        'KGE':   round(kge(y_true_np, y_pred_np), 4),
        'RMSE':  round(rmse(y_true_np, y_pred_np), 4),
        'PBIAS': round(pbias(y_true_np, y_pred_np), 2),
    }

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# BLOCK 7 │ Optuna Objectives 
# ─────────────────────────────────────────────────────────────────────────────
def objective_log_reg(trial):
    alpha     = trial.suggest_float("alpha",     1e-3, 20.0, log=True)
    log_shift = trial.suggest_float("log_shift", 1e-4,  1.0, log=True)
 
    y_tr_log = np.log(train_df[TARGET] + log_shift)
    model    = Ridge(alpha=alpha).fit(train_df[FEATURES], y_tr_log)
 
    preds_val = np.exp(model.predict(val_df[FEATURES])) - log_shift
    y_val_np  = val_df[TARGET].values
    denom     = np.sum((y_val_np - y_val_np.mean()) ** 2) + 1e-9
    return float(1 - np.sum((y_val_np - preds_val) ** 2) / denom)

def objective_ann(trial):
    h_size   = trial.suggest_int("hidden_size",  128,  512, step=64)
    n_layers = trial.suggest_int("n_layers",       2,    4)
    lr       = trial.suggest_float("lr",         1e-4, 5e-3, log=True)
    dropout  = trial.suggest_float("dropout",     0.0,  0.3)
    
    batch_sz = 32768   
 
    model   = wrap_model(SotaANN(len(FEATURES), h_size, n_layers, dropout))
    opt     = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched   = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=25)
    loss_fn = nn.HuberLoss()
 
    best_val, patience, wait = float('inf'), 5, 0
 
    for _ in range(25):
        model.train()
        perm = torch.randperm(len(X_tr_gpu), device=PRIMARY)
        for i in range(0, len(X_tr_gpu), batch_sz):
            idx = perm[i : i + batch_sz]
            opt.zero_grad(set_to_none=True)
            with autocast(device_type='cuda'):
                loss = loss_fn(model(X_tr_gpu[idx]), y_tr_gpu[idx])
            scaler_amp.scale(loss).backward()
            scaler_amp.step(opt)
            scaler_amp.update()
        sched.step()
 
        model.eval()
        with torch.no_grad(), autocast(device_type='cuda'):
            val_nse = nse(y_val_raw_gpu, descale(model(X_val_gpu)).flatten())
 
        val_loss = -val_nse
        if val_loss < best_val - 1e-4:
            best_val, wait = val_loss, 0
        else:
            wait += 1
            if wait >= patience:
                break
                
    del model, opt
    torch.cuda.empty_cache()
    return -best_val

def objective_lstm(trial):
    global _WINDOW_CACHE
    
    _Y_MEAN = torch.tensor(scaler_y.mean_,  device=PRIMARY, dtype=torch.float32)
    _Y_STD  = torch.tensor(scaler_y.scale_, device=PRIMARY, dtype=torch.float32)

    def _descale(p: torch.Tensor) -> torch.Tensor:
        return p * _Y_STD + _Y_MEAN

    window   = trial.suggest_int("window_size",  30,  90, step=30)
    h_size   = trial.suggest_int("hidden_size",  32, 128, step=32)
    n_layers = trial.suggest_int("n_layers",      1,   2)
    lr       = trial.suggest_float("lr",        1e-4, 1e-3, log=True)
    dropout  = trial.suggest_float("dropout",    0.0,  0.2)
    batch_sz = 1024

    Xtw_cpu, ytw_cpu = get_windows('train', window)
    Xvw_cpu, yvw_cpu = get_windows('val',   window)

    if len(Xtw_cpu) == 0:
        return float('-inf')

    model   = wrap_model(SotaAttentionLSTM(len(FEATURES), h_size, n_layers, dropout))
    opt     = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    loss_fn = nn.HuberLoss()

    try:
        for epoch in range(15):
            model.train()
            perm = torch.randperm(len(Xtw_cpu))
            for i in range(0, len(Xtw_cpu), batch_sz):
                idx = perm[i : i + batch_sz]
                bx = Xtw_cpu[idx].to(PRIMARY, non_blocking=True)
                by = ytw_cpu[idx].to(PRIMARY, non_blocking=True)
                opt.zero_grad(set_to_none=True)
                with autocast(device_type='cuda'):
                    loss = loss_fn(model(bx), by)
                scaler_amp.scale(loss).backward()
                scaler_amp.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler_amp.step(opt)
                scaler_amp.update()

            model.eval()
            with torch.no_grad():
                all_p = []
                for j in range(0, len(Xvw_cpu), batch_sz):
                    bxv = Xvw_cpu[j : j + batch_sz].to(PRIMARY)
                    all_p.append(model(bxv))
                preds_s = torch.cat(all_p)
                val_nse = nse(
                    _descale(yvw_cpu.to(PRIMARY)).flatten(),
                    _descale(preds_s).flatten()
                )

        return val_nse

    except torch.OutOfMemoryError:
        return -1.0
    finally:
        del model, opt
        torch.cuda.empty_cache()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# BLOCK 8 │ Hyperparameter Search (Ridge & ANN)
# ─────────────────────────────────────────────────────────────────────────────
N_TRIALS_LR   = 40
N_TRIALS_ANN  = 35
N_TRIALS_LSTM = 15

# ── Define the database path FIRST ──────────────────────────────────────────
DB = f"sqlite:///{PROJECT_ROOT}/Data_Files/floodnet_hpo.db"

print("🔎 [1/3] Log-Ridge baseline …")
study_lr = optuna.create_study(
    study_name="log_ridge",
    direction="maximize", 
    sampler=TPESampler(seed=SEED),
    storage=DB,
    load_if_exists=True
)
study_lr.optimize(objective_log_reg, n_trials=N_TRIALS_LR)
 
print("🔎 [2/3] Residual ANN …")
study_ann = optuna.create_study(
    study_name="res_ann",
    direction="maximize", 
    sampler=TPESampler(seed=SEED),
    storage=DB,
    load_if_exists=True
)
study_ann.optimize(objective_ann, n_trials=N_TRIALS_ANN)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# BLOCK 9 │ Memory Cleanup for LSTM
# ─────────────────────────────────────────────────────────────────────────────
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

for name in list(globals().keys()):
    if not name.startswith('_'):
        obj = globals()[name]
        if torch.is_tensor(obj) or (isinstance(obj, list) and len(obj) > 0 and torch.is_tensor(obj[0])):
            print(f"🧹 Deleting: {name}")
            del globals()[name]

gc.collect()
torch.cuda.empty_cache()

print(f"✅ VRAM Reset. Current Usage: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

# Ensure scaling tensors are rebuilt for LSTM search
Y_MEAN = torch.tensor(scaler_y.mean_,  device=PRIMARY, dtype=torch.float32)
Y_STD  = torch.tensor(scaler_y.scale_, device=PRIMARY, dtype=torch.float32)
print(f"✅ Y_MEAN={Y_MEAN.item():.4f}, Y_STD={Y_STD.item():.4f} — ready for LSTM search")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# BLOCK 10 │ Hyperparameter Search (LSTM)
# ─────────────────────────────────────────────────────────────────────────────
print("🔎 [3/3] Attention-LSTM …")
study_lstm = optuna.create_study(
    study_name="attn_lstm",
    direction="maximize", 
    sampler=TPESampler(seed=SEED),
    storage=DB,
    load_if_exists=True
)
study_lstm.optimize(objective_lstm, n_trials=N_TRIALS_LSTM)
 
print(f"\n{'─'*45}")
print(f"{'Model':<20} {'Val NSE':>10}")
print(f"{'─'*45}")
print(f"{'Log-Ridge':20} {study_lr.best_value:>10.4f}")
print(f"{'Res-ANN':20} {study_ann.best_value:>10.4f}")
print(f"{'Attn-LSTM':20} {study_lstm.best_value:>10.4f}")
print(f"{'─'*45}")

In [ ]:
# --- Row-level errors for Log-Ridge + Res-ANN (aligned to test_df rows) ---
examples = test_df[["deployment_id", "time", STORM_COL, TARGET]].copy()
examples["pred_log_ridge"] = lr_preds
examples["pred_res_ann"] = ann_preds

for model in ["log_ridge", "res_ann"]:
    examples[f"err_{model}"] = examples[TARGET] - examples[f"pred_{model}"]
    examples[f"abs_err_{model}"] = examples[f"err_{model}"].abs()

# "Got right" threshold (edit as needed)
HIT_TOL = 0.25  # inches
examples["hit_log_ridge"] = examples["abs_err_log_ridge"] <= HIT_TOL
examples["hit_res_ann"] = examples["abs_err_res_ann"] <= HIT_TOL

# Top examples
best_ann  = examples.nsmallest(30, "abs_err_res_ann")
worst_ann = examples.nlargest(30, "abs_err_res_ann")
best_lr   = examples.nsmallest(30, "abs_err_log_ridge")
worst_lr  = examples.nlargest(30, "abs_err_log_ridge")

# Storm-level failure cases
storm_fail_ann = (
    examples.groupby([STORM_COL, "deployment_id"], as_index=False)
    .agg(mae_ann=("abs_err_res_ann", "mean"),
         peak_obs=(TARGET, "max"),
         peak_pred_ann=("pred_res_ann", "max"),
         n_rows=("time", "count"))
    .sort_values("mae_ann", ascending=False)
)

# Save for review
out_dir = RESULTS_DIR / "error_examples"
out_dir.mkdir(parents=True, exist_ok=True)
examples.to_parquet(out_dir / "test_predictions_errors.parquet", index=False)
best_ann.to_csv(out_dir / "best_ann_examples.csv", index=False)
worst_ann.to_csv(out_dir / "worst_ann_examples.csv", index=False)
best_lr.to_csv(out_dir / "best_logridge_examples.csv", index=False)
worst_lr.to_csv(out_dir / "worst_logridge_examples.csv", index=False)
storm_fail_ann.head(100).to_csv(out_dir / "storm_failures_ann.csv", index=False)


In [ ]:
# %%─────────────────────────────────────────────────────────────────────────
# BLOCK 12 │ Visualisation
# ─────────────────────────────────────────────────────────────────────────────
# Re-load ANN for inference only — was deleted after Block 9 to free VRAM.
ann_eval = wrap_model(SotaANN(
    len(FEATURES), bp_ann['hidden_size'],
    bp_ann['n_layers'], bp_ann['dropout']
))
ann_eval.load_state_dict(torch.load(CHECKPOINT_DIR / "ann_final.pt")['model_state'])
ann_eval.eval()
 
with torch.no_grad():
    ann_preds_plot = descale(ann_eval(X_te_final_gpu)).cpu().numpy().flatten()
 
COLORS = {
    'obs':  '#1a1a2e',
    'lr':   '#e67e22',
    'ann':  '#8e44ad',
    'lstm': '#16a085',
    'grid': '#cccccc',
}
 
fig = plt.figure(figsize=(18, 16), dpi=150)
gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.48, wspace=0.32)
 
def pick_display_storm(min_rows=60):
    for sid in test_storms:
        n = (test_df[STORM_COL] == sid).sum()
        if n >= min_rows:
            return sid
    return test_storms[0]
 
focus_sid  = pick_display_storm()
storm_mask = test_df[STORM_COL].values == focus_sid
storm_obs  = test_df.loc[test_df[STORM_COL] == focus_sid, TARGET].values
storm_lr   = lr_preds[storm_mask]
storm_ann  = ann_preds_plot[storm_mask]
t_axis     = np.arange(len(storm_obs))

In [ ]:
# ── Panel A: ANN & Log-Ridge storm time-series ───────────────────────────────
ax1 = fig.add_subplot(gs[0, :])
ax1.fill_between(t_axis, 0, storm_obs, color=COLORS['obs'], alpha=0.12)
ax1.plot(t_axis, storm_obs, label='Observed (FloodNet)',
         color=COLORS['obs'], lw=2.5, alpha=0.95, zorder=3)
ax1.plot(t_axis, storm_lr,
         label=(f"Log-Ridge  NSE={metrics_df.loc['Log-Ridge','NSE']:.3f}  "
                f"KGE={metrics_df.loc['Log-Ridge','KGE']:.3f}"),
         color=COLORS['lr'], ls='--', lw=1.8, zorder=2)
ax1.plot(t_axis, storm_ann,
         label=(f"Res-ANN    NSE={metrics_df.loc['Res-ANN','NSE']:.3f}  "
                f"KGE={metrics_df.loc['Res-ANN','KGE']:.3f}"),
         color=COLORS['ann'], lw=2.0, zorder=2)
ax1.set_title(f"Storm Event Comparison — Storm ID: {focus_sid}",
              fontsize=13, fontweight='bold')
ax1.set_ylabel("Water Depth (in)", fontsize=11)
ax1.set_xlabel("Timestep (min)", fontsize=11)
ax1.legend(fontsize=9, framealpha=0.9)
ax1.grid(True, color=COLORS['grid'], alpha=0.5)

In [ ]:
# ── Panel B: LSTM storm segment ───────────────────────────────────────────────
DISP = min(800, len(lstm_preds))
ax2  = fig.add_subplot(gs[1, :])
t2   = np.arange(DISP)
ax2.fill_between(t2, 0, lstm_obs[:DISP], color=COLORS['obs'], alpha=0.12)
ax2.plot(t2, lstm_obs[:DISP], label='Observed (FloodNet)',
         color=COLORS['obs'], lw=2.5, alpha=0.95, zorder=3)
ax2.plot(t2, lstm_preds[:DISP],
         label=(f"Attn-LSTM  NSE={metrics_df.loc['Attn-LSTM','NSE']:.3f}  "
                f"KGE={metrics_df.loc['Attn-LSTM','KGE']:.3f}"),
         color=COLORS['lstm'], lw=2.0, zorder=2)
ax2.set_title("Attention-LSTM — Test Set Segment",
              fontsize=13, fontweight='bold')
ax2.set_ylabel("Water Depth (in)", fontsize=11)
ax2.set_xlabel("Timestep (min)", fontsize=11)
ax2.legend(fontsize=9, framealpha=0.9)
ax2.grid(True, color=COLORS['grid'], alpha=0.5)

In [ ]:
# ── Panel C: Scatter — Observed vs ANN Predicted ─────────────────────────────
ax3  = fig.add_subplot(gs[2, 0])
lim  = (min(y_te_raw.min(), ann_preds_plot.min()) * 0.95,
        max(y_te_raw.max(), ann_preds_plot.max()) * 1.05)
ax3.scatter(y_te_raw, ann_preds_plot, alpha=0.12, s=3,
            color=COLORS['ann'], rasterized=True)
ax3.plot(lim, lim, 'k--', lw=1.2, label='1:1 line')
ax3.set_xlim(lim); ax3.set_ylim(lim)
ax3.set_title("Res-ANN: Observed vs Predicted", fontsize=12, fontweight='bold')
ax3.set_xlabel("Observed (in)", fontsize=10)
ax3.set_ylabel("Predicted (in)", fontsize=10)
ax3.legend(fontsize=9); ax3.grid(True, color=COLORS['grid'], alpha=0.5)

In [ ]:
# ── Panel D: Grouped metric bars (NSE & KGE per model) ───────────────────────
ax4    = fig.add_subplot(gs[2, 1])
models = metrics_df.index.tolist()
x_pos  = np.arange(len(models))
bw     = 0.32
b1 = ax4.bar(x_pos - bw / 2, metrics_df['NSE'], bw,
             label='NSE', color='#2980b9', alpha=0.87)
b2 = ax4.bar(x_pos + bw / 2, metrics_df['KGE'], bw,
             label='KGE', color='#c0392b', alpha=0.87)
ax4.axhline(0,    color='black', lw=0.8, ls='--', alpha=0.5)
ax4.axhline(0.5,  color='green', lw=0.8, ls=':',  alpha=0.6, label='0.5 (satisfactory)')
ax4.axhline(0.65, color='green', lw=0.8, ls='--', alpha=0.4, label='0.65 (good)')
ax4.set_xticks(x_pos)
ax4.set_xticklabels(models, fontsize=10)
ax4.set_ylim(-0.15, 1.08)
ax4.set_ylabel("Score  (1 = perfect)", fontsize=10)
ax4.set_title("Model Performance: NSE & KGE", fontsize=12, fontweight='bold')
ax4.legend(fontsize=8, loc='lower right', framealpha=0.9)
ax4.grid(True, color=COLORS['grid'], alpha=0.5, axis='y')
 
for bar in list(b1) + list(b2):
    h = bar.get_height()
    ax4.text(bar.get_x() + bar.get_width() / 2, h + 0.012,
             f"{h:.3f}", ha='center', va='bottom', fontsize=8, fontweight='bold')
 
fig.suptitle(
    "NYC FloodNet — Flood Depth Prediction Model Shootout\n"
    "(Storm-aware CV  ·  Train / Val / Test  ·  Early Stopping  ·  2-GPU DataParallel)",
    fontsize=14, fontweight='bold', y=1.01
)
 
OUT_FIG = FIGURES_DIR / "flood_model_shootout.png"
plt.savefig(OUT_FIG, bbox_inches='tight', dpi=150)
plt.show()
print(f"✅ Figure saved → {OUT_FIG}")

In [ ]:
# %%─────────────────────────────────────────────────────────────────────────
# BLOCK 13 │ Summary
# ─────────────────────────────────────────────────────────────────────────────
print(f"""
╔══════════════════════════════════════════════════════╗
║           Training Complete — Output Summary         ║
╠══════════════════════════════════════════════════════╣
║  Checkpoints  → {str(CHECKPOINT_DIR):<36}║
║    ann_final.pt / lstm_final.pt / log_ridge_final.pkl║
║    scaler_X.pkl / scaler_y.pkl                      ║
║  Run log      → {str(log_path):<36}║
║  Figure       → {str(OUT_FIG):<36}║
╚══════════════════════════════════════════════════════╝
""")
 
 
# ── Entry point guard ─────────────────────────────────────────────────────────
# Keeps SLURM / module imports from triggering training on import.
if __name__ == "__main__":
    pass  # All blocks above run unconditionally in Jupyter.
          # Wrap in main() if converting to a pure .py script.